<a href="https://colab.research.google.com/github/pmadhyastha/INM434/blob/main/lab1_preprocessing_tokenization_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
__author__ = "Pranava Madhyastha"
__version__ = "INM434/IN3045 City, University of London, Spring 2026"

## Tokenisation: a brief overview

This is our first notebook for the module and we are going to focus on tokenisation. Please make sure to login to your respective google account to get started.

A tokenizer is used to split an input string into separate tokens or pieces, where each token represents a meaningful element in the input string. This is one of the most important steps in NLP. This step is also called text-preprocessing.

Tokenisation helps create a vocabulary of unique tokens overwhich we will run a variety of NLP algorithms. It mainly helps standardise the input data and help us squeeze in information from the long tail. It, in some cases, helps us to break the input text down into smaller, sometimes linguistically meaningful, units.

In this notebook, we will first build a simple tokeniser which only splits on white space. We will then look at a toy morphological analyser. We will then build a simplified version of subword based tokeniser.

---

How does this notebook work:

  * There will be some cells with "TODO": you will have to fill the code for the placeholder "YOUR CODE HERE".
  * You will also notice "ADVANCED TODO", this is for students who are indeed capable of writing

  * Each cell should take a few seconds to run, so if it is taking longer, there may be bug.

# Simple tokeniser

---



We will make use of the built in regular expression library in python to construct this tokeniser. Please refer to Chapter 2 (SLP: Jurafsky and Martin) references on regular expressions.

The tokeniser below will take split input text into separate words based on word boundaries. This is defined using the regular expression pattern "\b\w+\b". The re.findall() function is used to extract all the tokens from the text that match the pattern, and the list of tokens is returned.


In [ ]:
import re

def tokenise(text):
    # Define a regular expression pattern to split the text into tokens
    pattern = r'\b\w+\b'

    # Use the re.findall() function to extract all the tokens from the text
    tokens = re.findall(pattern, text)

    # Return the list of tokens
    return tokens

# Sample input for the tokeniser
text = "This is IN 3045/INM 434 Natural Language Procssing. This is some sample text for tokeniser"
tokens = tokenise(text)
print(tokens)

**What does '\b\w+\b' mean?**

The regular expression \b\w+\b is used to find whole words in a string. Let's see it in detail:

\b: This is a word boundary. It matches the position between a word character (like a letter, number, or underscore) and a non-word character (like a space, punctuation, or the start/end of the string).
\w: This matches any word character (alphanumeric characters plus underscore). This includes a-z, A-Z, 0-9, and _.
+: This is a quantifier that means "one or more" of the preceding character or group. So, \w+ means one or more word characters.
Combined, \b\w+\b finds sequences of one or more word characters that are surrounded by word boundaries, effectively extracting whole words.

See Chapter 2 for more details, but also refer: https://docs.python.org/3/library/re.html

---
# Building a Sentiment Classifier <a name="classifier"></a>

Let's build a sentiment classifier that predicts whether a movie review is **positive** or **negative** -- this is one of the classic NLP tasks.

## Our Plan is:

1. **Tokenise**: Input the text by tokenising the text
2. **Vectorise**: Convert the elements of the text into a bag-of-words representation
3. **Train**: Learn which words indicate positive/negative sentiment
4. **Predict**: Classify new reviews

In [ ]:
import re
import numpy as np
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

# Our training data: movie reviews with sentiment labels
training_data = [
    # Positive reviews
    ("This movie is amazing and wonderful", "positive"),
    ("I loved every minute of this film", "positive"),
    ("Brilliant acting and great story", "positive"),
    ("One of the best movies I have seen", "positive"),
    ("Absolutely fantastic and entertaining", "positive"),
    ("A beautiful and moving experience", "positive"),
    ("Highly recommend this masterpiece", "positive"),
    ("Excellent film with superb performances", "positive"),
    ("I really enjoyed this movie", "positive"),
    ("What a great and fun movie", "positive"),

    # Negative reviews
    ("This movie is terrible and boring", "negative"),
    ("I hated every minute of this film", "negative"),
    ("Awful acting and stupid story", "negative"),
    ("One of the worst movies ever made", "negative"),
    ("Absolutely dreadful waste of time", "negative"),
    ("A painful and tedious experience", "negative"),
    ("Do not waste your money on this", "negative"),
    ("Horrible film with bad performances", "negative"),
    ("I really disliked this movie", "negative"),
]


Please write code to list the number of training examples in total, with the number of positive and negative examples.

In [ ]:
print(f"Training data: {len(training_data)} reviews")
print(f"  - Positive: {sum(1 for _, label in training_data if label == 'positive')}")
print(f"  - Negative: {sum(1 for _, label in training_data if label == 'negative')}")

## Simple Tokenizer

# First, we need to split text into words. Let's start with a simple approach:

In [ ]:
def simple_tokenize(text):

    # TODO - write code to convert to lowercase
    text = text.lower()


    # Write code to Split on word boundaries, keep only alphanumeric
    tokens = re.findall(r'\b\w+\b', text)

    return tokens

## Building Vocabulary

We need to know all unique words in our training data -- this is the VOCABULARY of our model

In [ ]:
def build_vocabulary(data, tokenize_fn):

    # we will use COUNTER class from python -- https://docs.python.org/3/library/collections.html
    word_counts = Counter()

    for text, label in data:
        tokens = tokenize_fn(text)
        word_counts.update(tokens)

    # TODO: What is counter doing?

    # Create word-to-index mapping
    vocab = {word: idx for idx, word in enumerate(sorted(word_counts.keys()))}

    return vocab, word_counts

# Build vocabulary
vocab, word_counts = build_vocabulary(training_data, simple_tokenize)


TODO: Write code to print the size of the vocabulary and also list the most frequent to least frequent words.

In [ ]:

print(f"Vocabulary size: {len(vocab)} unique words")
print(f"\nMost common words:")
for word, count in word_counts.most_common(10):
    print(f"  '{word}': {count}")

## Bag-of-Words Representation

Convert each document to a vector (in our case an array) of word counts

In [ ]:
def text_to_bow(text, vocab, tokenize_fn):
    # Initialize vector of zeros
    bow = np.zeros(len(vocab))

    # Count words
    tokens = tokenize_fn(text)
    for token in tokens:
        if token in vocab:
            bow[vocab[token]] += 1

    return bow


example  = "This movie is "
bow_vector = text_to_bow(example, vocab, simple_tokenize)

TODO: Print out the bag of word vector. Also, print out all the "tokens" with non-zero entries.

In [ ]:
print(f"Text: '{example}'")
print(f"BoW vector shape: {bow_vector.shape}")
print(f"Non-zero entries:")
for word, idx in vocab.items():
    if bow_vector[idx] > 0:
        print(f"  {word}: {bow_vector[idx]}")

## Now Let us train a Simple Classifier

We will now use a very simple classifier -- the Naive Bayes classifier which is a simple but effective classifier for text.

The idea: estimate P(word | positive) and P(word | negative), then use Bayes' rule.

Essentially we are seeing If a review is positive, will it have the word 'good'?. Then, we ask, if I indeed see the word 'good', what is the probability the review is positive?. It allows us to update our belief about the review's sentiment as we read each word -- this is a fairly naive way of approaching language -- here we don't care about the rich structure!

In [ ]:
class NaiveBayesClassifier:


    def __init__(self, tokenize_fn):
        self.tokenize_fn = tokenize_fn
        self.vocab = None
        self.word_probs = {}  # P(word | class)
        self.class_probs = {}  # P(class)

    def train(self, data):

        # Build vocabulary
        self.vocab, word_counts = build_vocabulary(data, self.tokenize_fn)

        # Count words per class -- TODO: Why do we need this?
        class_word_counts = defaultdict(Counter)
        class_doc_counts = Counter()

        for text, label in data:
            tokens = self.tokenize_fn(text)
            class_word_counts[label].update(tokens)
            class_doc_counts[label] += 1

        # Calculate probabilities with smoothing -- this is done to prevent division by zero!
        total_docs = len(data)
        vocab_size = len(self.vocab)

        for label in class_doc_counts:
            # P(class)
            self.class_probs[label] = class_doc_counts[label] / total_docs

            # P(word | class) with Laplace smoothing
            total_words = sum(class_word_counts[label].values())
            self.word_probs[label] = {}

            for word in self.vocab:
                count = class_word_counts[label][word]
                # Laplace smoothing -- add 1 to avoid zero probabilities
                self.word_probs[label][word] = (count + 1) / (total_words + vocab_size)

        print(f"Trained on {total_docs} documents")
        print(f"Vocabulary size is: {vocab_size}")
        print(f"Total number of classes: {list(class_doc_counts.keys())}")

    def predict(self, text, return_probs=False):

        tokens = self.tokenize_fn(text)

        # Calculate log probability for each class
        log_probs = {}

        for label in self.class_probs:
            # Start with log P(class)
            log_prob = np.log(self.class_probs[label])

            # Add log P(word | class) for each word
            for token in tokens:
                if token in self.word_probs[label]:
                    log_prob += np.log(self.word_probs[label][token])
                # Ignore unknown words (or could use a small probability)

            log_probs[label] = log_prob

        # Predict class with highest probability
        predicted = max(log_probs, key=log_probs.get)

        if return_probs:
            # Convert to actual probabilities
            max_log = max(log_probs.values())
            probs = {k: np.exp(v - max_log) for k, v in log_probs.items()}
            total = sum(probs.values())
            probs = {k: v/total for k, v in probs.items()}
            return predicted, probs

        return predicted

    def get_important_words(self, n=10):

        important = {}

        for label in self.class_probs:
            other_label = [l for l in self.class_probs if l != label][0]

            ratios = []
            for word in self.vocab:
                ratio = self.word_probs[label][word] / self.word_probs[other_label][word]
                ratios.append((word, ratio))

            ratios.sort(key=lambda x: x[1], reverse=True)
            important[label] = ratios[:n]

        return important

# Let us now train our classifier
classifier = NaiveBayesClassifier(simple_tokenize)
classifier.train(training_data)

## So what has it learned?

Let us see which words are most indicative of positive vs negative sentiment?

In [ ]:
important_words = classifier.get_important_words(10)


TODO: Print out the most important words and their corresponding ratios

In [ ]:
print("Most indicative words:")
print("=" * 50)

for label, words in important_words.items():
    print(f"\n{label.upper()}:")
    for word, ratio in words:
        bar = "---" * int(min(ratio, 20))
        print(f"  {word:<15} {ratio:>6.2f}x  {bar}")

---
# Does it work? <a name="test"></a>

Let's test our classifier on some examples:

In [ ]:

test_examples = [
    "This movie is amazing",
    "I loved this film",
    "Terrible and boring",
    "The worst movie ever",
    "A great and entertaining film",
    "Awful waste of time",
]

print("Testing our classifier:")
print("=" * 60)

for text in test_examples:
    pred, probs = classifier.predict(text, return_probs=True)
    confidence = probs[pred] * 100
    emoji = "correct" if pred == "positive" else "incorrect"
    print(f"{emoji} '{text}'")
    print(f"   --> {pred} ({confidence:.1f}% confident)\n")

# How did it do?

Our simple bag-of-words classifier correctly identifies sentiment for straightforward cases.

Even this simple approach works because:
- Positive reviews tend to use words like "amazing", "loved", "great"
- Negative reviews tend to use words like "terrible", "awful", "worst"

Now let's look at some tricky cases!

In [ ]:

tricky_examples = [
    # Negation ones
    ("This movie is not good", "negative"),
    ("I don't like this film", "negative"),
    ("Not bad at all", "positive"),
    ("I didn't hate it", "positive"),

    # Say there is some sarcasm or contrasting bits
    ("Oh great, another terrible movie", "negative"),
    ("The acting was good but the story was awful", "mixed"),

    # Say we have words not seen in the data ?
    ("This film is phenomenal", "positive"),
    ("Utterly abysmal", "negative"),
    ("A cinematic triumph", "positive"),

    # Intensity bits
    ("This movie is good", "positive"),
    ("This movie is REALLY good", "positive"),
    ("This movie is soooo good!!!", "positive"),

    # Word variations
    ("I loved it", "positive"),
    ("I'm loving it", "positive"),
    ("Lovable characters", "positive"),
]

print("Testing on TRICKY examples:")
print("=" * 70)

correct = 0
wrong = 0

for text, expected in tricky_examples:
    pred, probs = classifier.predict(text, return_probs=True)
    confidence = probs[pred] * 100

    if expected == "mixed":
        status = "dont know"  # Can't really be right or wrong
    elif pred == expected:
        status = "correct"
        correct += 1
    else:
        status = "incorrect"
        wrong += 1

    print(f"{status} '{text}'")
    print(f"   Expected: {expected}, Got: {pred} ({confidence:.1f}%)\n")

print("=" * 70)
print(f"Results: {correct} correct, {wrong} wrong (excluding 'mixed')")

# Advanced TODO:
Can we come up with a negation aware tokenizer? That is -- say you tokenize such that "the movie is not good" is tokenized as ["the", "movie", "is", "not",  "NOT_good"].

In [ ]:
# TODO
# 1. Create a new classifier with tokenize_with_negation -- that means you
#    would have to create a function called tokenize_with_negation which
#    captures negation aware tokenization.
# 2. Train it on training_data
# 3. Test on the negation examples from tricky_examples

classifier_negation = NaiveBayesClassifier(tokenize_with_negation)
classifier_negation.train(training_data)

# Test on negation examples
negation_tests = [
    ("This movie is not good", "negative"),
    ("I don't like this film", "negative"),
    ("Not bad at all", "positive"),
]

print("Testing negation-aware tokenizer:")
for text, expected in negation_tests:
    tokens = tokenize_with_negation(text)
    pred = classifier_negation.predict(text)
    status = "correct" if pred == expected else "incorrect"
    print(f"{status} '{text}'")
    print(f"   Tokens: {tokens}")
    print(f"   Expected: {expected}, Got: {pred}\n")

In [ ]:
def tokenize_with_negation(text):
    """Tokenize and attach negation to following word."""
    text = text.lower()
    # Handle contractions
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'m", " am", text)
    text = re.sub(r"'s", " is", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'ll", " will", text)

    tokens = re.findall(r'\b\w+\b', text)

    # Attach NOT_ prefix to words following negation
    result = []
    negate_next = False
    for token in tokens:
        if token in ['not', 'no', 'never', 'neither']:
            negate_next = True
            result.append(token)
        elif negate_next:
            result.append('NOT_' + token)
            negate_next = False
        else:
            result.append(token)

    return result

def tokenize_characters(text):
    """Character-level tokenization."""
    return list(text.lower())




---
# Better Tokenization with BPE <a name="part5"></a>

**Byte-Pair Encoding (BPE)** learns a vocabulary from data that balances:
- Small vocabulary size
- Meaningful units (not just characters)
- No OOV problem (can always fall back to characters)

This is what GPT, BERT, and most modern LLMs use!


Sennrich, Rico, Barry Haddow, and Alexandra Birch. "Neural Machine Translation of Rare Words with Subword Units." Proceedings of the 54th Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers). Vol. 1. 2016. http://www.aclweb.org/anthology/P16-1162

Main source: https://github.com/rsennrich/subword-nmt

Below is a simple version of the algorithm.

Two stages: token learner and token segmenter.

Let us first look at token learner, this involves:

*  computing the frequencies of all words in a corpus (we do it synthetically here)
*  start with characters as the basic vocab (characters seen in the corpus)
* to obtain vocabulary of n-merge operations:
    - Obtain most frequent pairs of characters in the corpus
    - add the pair to the list of merges
    - add merged characters to the vocab
* iterate n times

The code below performs this operation.

In [ ]:
import collections

def compute_pair_frequencies(vocab):

    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq
    return pairs


def merge_vocab(pair, vocab):

    new_vocab = {}
    bigram = re.escape(' '.join(pair))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')

    for word in vocab:
        new_word = pattern.sub(''.join(pair), word)
        new_vocab[new_word] = vocab[word]

    return new_vocab


def train_bpe(corpus_words, num_merges):

    # Initialize: split each word into characters + end marker
    vocab = {}
    for word, freq in corpus_words.items():
        symbols = ' '.join(list(word)) + ' </w>'
        vocab[symbols] = freq

    bpe_codes = {}

    print("BPE Training")
    print("=" * 60)
    print(f"Initial vocabulary: {vocab}\n")

    for i in range(num_merges):
        pairs = compute_pair_frequencies(vocab)

        if not pairs:
            print("No more pairs to merge!")
            break

        # Find most frequent pair
        best_pair = max(pairs, key=pairs.get)
        best_freq = pairs[best_pair]

        # Merge these!
        vocab = merge_vocab(best_pair, vocab)
        bpe_codes[best_pair] = i

        print(f"Merge {i+1}: '{best_pair[0]}' + '{best_pair[1]}' -> '{best_pair[0]+best_pair[1]}' (freq: {best_freq})")

    print(f"\nFinal vocabulary: {vocab}")

    return bpe_codes, vocab

Now we need to train it -- that is, it learns from data!

In [ ]:
# Train BPE on a small corpus
corpus_words = {
    'low': 5,
    'lower': 2,
    'newest': 6,
    'widest': 3,
    'new': 4,
}

bpe_codes, final_vocab = train_bpe(corpus_words, num_merges=10)

In [ ]:
def bpe_encode(word, bpe_codes):

    # Start with characters + end marker
    word = tuple(word) + ('</w>',)

    while len(word) > 1:
        # Find all adjacent pairs
        pairs = [(word[i], word[i+1]) for i in range(len(word)-1)]

        # Find pair with lowest merge rank (earliest learned)
        min_pair = None
        min_rank = float('inf')

        for pair in pairs:
            if pair in bpe_codes and bpe_codes[pair] < min_rank:
                min_pair = pair
                min_rank = bpe_codes[pair]

        # If no pair can be merged, we're done
        if min_pair is None:
            break

        # Merge all occurrences of min_pair
        first, second = min_pair
        new_word = []
        i = 0

        while i < len(word):
            if i < len(word) - 1 and word[i] == first and word[i+1] == second:
                new_word.append(first + second)
                i += 2
            else:
                new_word.append(word[i])
                i += 1

        word = tuple(new_word)

    # Clean up end marker for display
    result = []
    for token in word:
        if token == '</w>':
            continue
        elif token.endswith('</w>'):
            result.append(token[:-4])
        else:
            result.append(token)

    return tuple(result)


# Test BPE encoding
test_words = ['low', 'lower', 'lowest', 'new', 'newer', 'newest', 'widest', 'unknown']

print("\nBPE Encoding Results:")
print("=" * 40)
for word in test_words:
    encoded = bpe_encode(word, bpe_codes)
    print(f"{word:<12} -> {encoded}")

Okay, observe something interesting (even in this small set of examples). The BPE tokenizer has learned to split the words into:
- `est` (superlative suffix)
- `er` (comparative suffix, eventually)
- `low`, `new` (common roots)

This happened automatically from frequency, not from any linguistic rules!

But, this may not be always true, as you see in the outputs!  

ADVANCED TODO: Train BPE on the full corpus above.

In [ ]:
# perhaps easy to write a function!

def get_corpus_word_frequencies(data):
    """Extract word frequencies from the training corpus."""
    word_freq = Counter()
    for text, label in data:
        words = simple_tokenize(text)
        word_freq.update(words)
    return dict(word_freq)

# Get word frequencies from our training data
corpus_word_freq = get_corpus_word_frequencies(training_data)


# Now let us train BPE on the full corpus

corpus_bpe_codes, corpus_final_vocab = train_bpe(corpus_word_freq, num_merges=50)


# Get all unique tokens from final vocabulary
bpe_tokens = set()
for word in corpus_final_vocab:
    for token in word.split():
        bpe_tokens.add(token)

